# Tests des fonctions de calcul de la saturation

In [1]:
import datetime
from datetime import timedelta

import pandas as pd
from pandas import NamedAgg
from saturation_image_quali import (
    filter_sessions_duration,
    hourly_maximum,
    hysteresis,
    maxi_duration,
    to_sampled_sessions,
    to_sampled_state_grp,
    to_sampled_state_poc,
    to_sampled_statuses,
    to_state_grp_d,
    to_state_poc_d,
)

# from saturation import (
    # hysteresis,
    # to_sampled_sessions,
    # to_sampled_state_grp,
    # to_sampled_state_pdc,
    # to_sampled_statuses,
#)

## Test échantillonage des sessions

In [2]:
samples_per_day = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = ['p1', 'p2', 'p3']
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1] -> [1.1, 2, 2]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6] -> [1.5, 2, 3.1, 2.6]
sessions = to_sampled_sessions(filter_sessions_duration(test), init, timestamp, samples_per_day)

assert len(sessions) == 72
assert sessions.iloc[0]['occupation_pdc'] == 'occupe'
assert sessions.iloc[1]['occupation_pdc'] == 'occupe'
assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'
assert sessions.iloc[34]['occupation_pdc'] == 'occupe'


In [3]:
sessions = to_sampled_sessions(filter_sessions_duration(test, min_duration=datetime.timedelta(hours=1.2)), init, timestamp, samples_per_day)
assert sessions.iloc[1]['occupation_pdc'] == 'f_libre'

# sessions

In [4]:
sessions = to_sampled_sessions(filter_sessions_duration(test, max_duration=datetime.timedelta(hours=3)), init, timestamp, samples_per_day)
assert sessions.iloc[34]['occupation_pdc'] == 'f_libre'

# sessions

In [5]:
samples_per_day = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [6.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']}) 
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 6.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]

sessions = to_sampled_sessions(filter_sessions_duration(test), init, timestamp, samples_per_day)

assert sessions.iloc[5]['occupation_pdc'] == 'occupe'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

# sessions

## Test échantillonage des statuts

In [6]:
samples_per_day = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
values = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in values],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'occupation_pdc':['occupe', 'libre', 'libre', 'occupe', 
                                                      'occupe', 'libre', 'libre'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = ['p1', 'p2', 'p3']
init_start = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
init_end = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc})
init = pd.concat([init_start, init_end])
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day)
assert len(statuses) == 72
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day, datetime.timedelta(hours=1.9))
assert len(statuses) == 72
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'en_service'

#statuses

In [7]:
samples_per_day = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
values = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in values],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'occupation_pdc':['occupe', 'libre', 'libre', 'occupe', 
                                                      'occupe', 'libre', 'libre'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init_start = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
init_end = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc})
init = pd.concat([init_start, init_end])
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day)
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day, datetime.timedelta(hours=1.9))
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'en_service'

# statuses

In [8]:
samples_per_day = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
values = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in values],
                      'etat_pdc':['en_service', 'hors_service', 'inconnu', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'occupation_pdc':['occupe', 'libre', 'inconnu', 'occupe', 
                                                      'occupe', 'libre', 'libre'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init_start = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
init_end = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'occupation_pdc':['libre'] * len(pdc),
                      'id_pdc_itinerance': pdc})
init = pd.concat([init_start, init_end])
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day)
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, samples_per_day, datetime.timedelta(hours=1.9))
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'

# statuses

In [9]:
from datetime import date, timedelta

day = date(2026,7, 14)
timestamp = pd.Timestamp('2026-07-14T00:00:00+02:00')
samples_per_day = 288
min_duration = timedelta(minutes=24 * 60 / samples_per_day)

all_pocs = pd.Series(['p1', 'p2'])
init_start_statuses = pd.DataFrame(
    {
        "horodatage": [timestamp + pd.Timedelta(days=-1)] * len(all_pocs),
        "etat_pdc": ["en_service"] * len(all_pocs),
        "occupation_pdc": ["libre"] * len(all_pocs),
        "id_pdc_itinerance": all_pocs,
    }
)
init_end_statuses = pd.DataFrame(
        {
            "horodatage": [timestamp + pd.Timedelta(days=1)] * len(all_pocs),
            "etat_pdc": ["en_service"] * len(all_pocs),
            "occupation_pdc": ["libre"] * len(all_pocs),
            "id_pdc_itinerance": all_pocs,
        }
    )
init_statuses = pd.concat([init_start_statuses, init_end_statuses])
values = [10.45, 10.5]
statuses = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in values],
                      'etat_pdc':['hors_service', 'en_service'],
                      'occupation_pdc':['inconnu', 'libre'],
                      'id_pdc_itinerance': ['p1', 'p1']})
sampled_statuses = to_sampled_statuses(
    statuses, init_statuses, timestamp, samples_per_day, min_duration=min_duration
)
assert sampled_statuses['etat_pdc'].eq('en_service').all()
assert sampled_statuses['occupation_pdc'].eq('libre').all()

## Test assemblage des sessions et des statuts

In [23]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'occupe']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'libre', 'occupe', 'libre', 'occupe', 'libre','libre', 'occupe', 'occupe'],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})

merged = to_sampled_state_poc(sessions, status)
assert merged['pseudo_libre'].eq([False]*9).all()
assert merged['pseudo_occupe'].eq([False]*9).all()

status['id_pdc_itinerance'] = ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4']
merged = to_sampled_state_poc(sessions, status)
assert merged['pseudo_libre'].sum() == 1
assert merged['pseudo_occupe'].sum() == 1

status['id_pdc_itinerance'] = ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3']
sessions.loc[2, 'occupation_pdc'] = 'f_libre'
merged = to_sampled_state_poc(sessions, status)
assert merged['pseudo_libre'].sum() == 0
assert merged['pseudo_occupe'].sum() == 1

sessions.loc[2, 'occupation_pdc'] = 'occupe'
sessions.loc[3, 'occupation_pdc'] = 'occupe'
merged = to_sampled_state_poc(sessions, status)
assert merged['pseudo_libre'].sum() == 1
assert merged['pseudo_occupe'].sum() == 0

samples_per_day = 24
state_d = to_state_poc_d(merged, samples_per_day)
assert state_d.iloc[0].eq(pd.Series(['p1', 120.0, 60.0, 60.0, 0.0, 0.0, 0.0])).all()
assert state_d.iloc[1].eq(['p2', 120.0, 60.0, 60.0, 0.0, 60.0, 0.0]).all()
assert state_d.iloc[2].eq(['p3', 120.0, 60.0, 0.0, 60.0, 0.0, 0.0]).all()

#sessions

AssertionError: 

In [28]:
assert list(state_d.iloc[0].values) == ['p1', 120.0, 60.0, 60.0, 0.0, 0.0, 0.0]

In [25]:
pd.Series(['p1', 120.0, 60.0, 60.0, 0.0, 0.0, 0.0])

0       p1
1    120.0
2     60.0
3     60.0
4      0.0
5      0.0
6      0.0
dtype: object

In [11]:
#state_d

## Test état global échantillonné d'un groupement de pdc

In [12]:
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [timestamp + pd.Timedelta(hours=val) for val in [0, 1, 2, 0, 1, 2, 0, 1, 2]],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service'],
                     'pseudo_libre' : [False, False, False, False, False, False, True, False, False],
                     'pseudo_occupe' : [False, False, False, False, False, False, False, True, False],})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
sampled_state_station = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2, add_full_use=True, add_latency=True)
sampled_state_station

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,s1,2025-04-25 00:00:00+02:00,1,0,1,2,0,0,2,False,False,True,False,False,True,3
1,s1,2025-04-25 01:00:00+02:00,1,1,0,2,0,0,2,False,False,True,True,False,False,5
2,s1,2025-04-25 02:00:00+02:00,1,0,1,2,0,0,2,False,False,True,False,False,True,3
3,s2,2025-04-25 00:00:00+02:00,0,0,1,0,1,0,1,False,True,False,False,False,False,2
4,s2,2025-04-25 01:00:00+02:00,1,0,0,1,0,1,1,False,False,True,True,False,False,5
5,s2,2025-04-25 02:00:00+02:00,0,1,0,1,0,0,1,True,False,True,False,False,False,1


In [13]:
hourly_max_pu = hourly_maximum(sampled_state_station, "id_station_itinerance", "periode", "pu", 12)
assert list(hourly_max_pu['index_state_grp']) == [2, 5]
assert list(hourly_max_pu['pu_max']) == [3.0, 2.0]
hourly_max_pu

,id_station_itinerance,index_state_grp,pu_max
0,s1,2,3.0
1,s2,5,2.0


In [14]:
max_duration = maxi_duration(sampled_state_station, "id_station_itinerance", "periode", "pu")
assert list(max_duration['first']) == [pd.Timestamp('2025-04-25 00:00:00+02:00'), pd.Timestamp('2025-04-25 01:00:00+02:00')]
assert list(max_duration['duration']) == [3, 2]
max_duration

,id_station_itinerance,valid_group,first,last,duration
0,s1,1,2025-04-25 00:00:00+02:00,2025-04-25 02:00:00+02:00,3
1,s2,2,2025-04-25 01:00:00+02:00,2025-04-25 02:00:00+02:00,2


## Test de la pleine utilisation

In [15]:
# une station
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2',
                                           'p3', 'p3', 'p3', 'p3',
                                           'p4', 'p4', 'p4', 'p4',
                                           'p5', 'p5', 'p5', 'p5',
                                           'p6', 'p6', 'p6', 'p3'],
                     'periode' : [timestamp + pd.Timedelta(hours=val) for val in [0, 1, 2, 3] * 6],
                     # 'periode' : timestamp,
                     'state' : ['libre', 'occupe', 'occupe', 'occupe'] * 6,
                     'pseudo_libre' : [True, True, False, False] * 6,
                     'pseudo_occupe' : [False, True, True, False] * 6})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
                         'id_station_itinerance': ['s1'] * 6}) 
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2)
assert res['sature'][3]
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True, add_latency=True)
assert (res['sature'] == res['pu']).all()
test.loc[2,'state'] = 'libre'
test.loc[3,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True, add_latency=True)
assert not res['sature'][2]
assert res['pu'][2]
assert res['pu'][3]
test.loc[6,'state'] = 'libre'
test.loc[7,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True, add_latency=True)
assert res['pu'][2]
assert not res['pu'][3]
res

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,s1,2025-04-25 00:00:00+02:00,0,0,6,3,6,0,6,False,True,False,False,False,False,2
1,s1,2025-04-25 01:00:00+02:00,6,0,0,6,6,6,6,False,False,True,True,False,False,5
2,s1,2025-04-25 02:00:00+02:00,4,0,2,6,0,6,6,False,False,True,False,False,True,3
3,s1,2025-04-25 03:00:00+02:00,4,0,2,4,0,0,6,False,False,False,False,False,True,3


In [16]:
res_d = to_state_grp_d(res, 'id_station_itinerance', 24 )
assert res_d['pu_max'].values[0] == 60.0
assert res_d['sature_max'].values[0] == 60.0
assert list(res_d[["hs", "inactif", "pu_cum", "sature_cum", "surcharge", "actif"]].values[0]) == list((res[["hs", "inactif", "pu", "sature", "surcharge", "actif"]].sum() * 60.0).values)
res_d

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
0,s1,6,0.0,60.0,120.0,60.0,0.0,120.0,60.0,60.0,120.0


In [17]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre'],
                     'pseudo_libre' : [True, True, False, False] * 3,
                     'pseudo_occupe' : [False, True, True, False] * 3})                                
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,s1,0,1,0,1,1,1,1,2,False,False,False,False,False,True,3
1,s1,1,0,1,1,0,1,1,2,False,True,False,False,False,False,2
2,s1,2,2,0,0,2,1,1,2,False,False,False,True,False,False,5
3,s1,3,1,1,0,1,1,1,2,False,False,False,True,False,False,5
4,s1,4,0,2,0,0,1,1,2,True,False,False,False,False,False,1
5,s1,5,0,0,2,0,1,1,2,False,True,False,False,False,False,2


## test de l'hysteresis

In [18]:
# seuil à 6 et 9
serie = pd.Series([1, 2, 5, 7, 5, 8, 10, 12, 8, 11, 8, 5, 7, 2])
res = hysteresis(serie, 6, 9)
assert res[6:10].all() == True
assert res[:5].all() == False

## test maxi duration 

In [19]:
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
periodes = [1, 2, 3, 4, 5, 6, 7, 8, 9] * 2

df = pd.DataFrame({'occupe': [False, True, True, False, True, True, True, False, True,
                              False, True, True, True, True, False, True, False, True],
                   'station': ['s1'] * 9 + ['s2'] * 9,
                   'periode': [timestamp + pd.Timedelta(hours=per) for per in periodes]})
                   #'periode': periodes})

pu_duration = maxi_duration(df, 'station', 'periode', 'occupe')
assert list(pu_duration['duration']) == [3, 4]
pu_duration

,station,valid_group,first,last,duration
0,s1,4,2025-04-25 05:00:00+02:00,2025-04-25 07:00:00+02:00,3
1,s2,2,2025-04-25 02:00:00+02:00,2025-04-25 05:00:00+02:00,4


In [20]:
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
periodes = [1, 2, 3, 4, 5, 6, 7, 8, 9] * 2

df = pd.DataFrame({'occupe': [False, True, False, True, False, True, False, True, False] + [False] * 9,
                   'station': ['s1'] * 9 + ['s2'] * 9,
                   'periode': [timestamp + pd.Timedelta(hours=per) for per in periodes]})

pu_duration = maxi_duration(df, 'station', 'periode', 'occupe')
assert pu_duration['duration'].iloc[0] == 1
assert len(pu_duration) == 1
pu_duration

,station,valid_group,first,last,duration
0,s1,2,2025-04-25 02:00:00+02:00,2025-04-25 02:00:00+02:00,1


## test ajout du nombre de sessions et de la puissance

In [21]:
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')

sessions = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                          'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                          'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2'],
                          'energy': [10, 20, 30, 40, 50, 60, 70]})
states = pd.DataFrame({'id_pdc_itinerance': ['p0', 'p2', 'p3'],
                       'libre': [10, 20, 30],
                       'occupe': [5, 10, 15]})
infos_poc = sessions.groupby('id_pdc_itinerance').agg(
                sessions_nb=NamedAgg("energy", "count"),
                energy_cum=NamedAgg("energy", "sum")
            ).reset_index()

full_states = pd.merge(states, infos_poc, on='id_pdc_itinerance', how='left').fillna(0)

assert full_states.loc[full_states['id_pdc_itinerance'] == 'p2', 'sessions_nb'].values[0] == 4
assert full_states.loc[full_states['id_pdc_itinerance'] == 'p2', 'energy_cum'].values[0] == 170

full_states

,id_pdc_itinerance,libre,occupe,sessions_nb,energy_cum
0,p0,10,5,0.0,0.0
1,p2,20,10,4.0,170.0
2,p3,30,15,0.0,0.0
